# Week 8 — Logistic regression: controlled comparison

### Deliverable
- Baseline vs extended model + odds ratios + interpretation


In [1]:
# Core imports (kept minimal for beginners)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 120)

# Dataset URL (UCI Heart Disease - Cleveland)
UCI_URL = "https://www.archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"

# Column names for processed.cleveland.data (14 columns commonly used in teaching)
COLS = ["age","sex","cp","trestbps","chol","fbs","restecg","thalach",
        "exang","oldpeak","slope","ca","thal","num"]


In [2]:
import statsmodels.api as sm


In [3]:
import ssl
import io
import urllib.request # Added this import

def load_raw():
    # Create an unverified SSL context to bypass certificate verification
    ctx = ssl.create_default_context()
    ctx.check_hostname = False
    ctx.verify_mode = ssl.CERT_NONE

    # Open the URL with the unverified context
    with urllib.request.urlopen(UCI_URL, context=ctx) as url_response:
        # Read the content and decode it
        s = url_response.read().decode('utf-8')

    # Use io.StringIO to make the string behave like a file for pandas.read_csv
    df_raw = pd.read_csv(io.StringIO(s), header=None, names=COLS)
    return df_raw

def coerce_types(df_raw):
    # Missing values are sometimes encoded as "?"
    df = df_raw.replace("?", np.nan).copy()

    # Convert numeric-looking columns
    numeric_cols = ["age","trestbps","chol","thalach","oldpeak","ca","thal","num"]
    for c in numeric_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # Binary target: disease present if num > 0 (UCI convention)
    df["disease"] = (df["num"] > 0).astype("Int64")
    return df

df_raw = load_raw()
df = coerce_types(df_raw)

df.head()


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,num,disease
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0,0,0
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,2,1
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1,1
3,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0,0,0
4,41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0,0,0


In [4]:
def clean_heart(df_raw):
    df = df_raw.replace("?", np.nan).copy()
    numeric_cols = ["age","trestbps","chol","thalach","oldpeak","ca","thal","num"]
    for c in numeric_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df["disease"] = (df["num"] > 0).astype(int)
    return df

df_clean = clean_heart(df_raw)

base_features = ["age","sex"]
extra_features = ["thalach","oldpeak","exang"]  # TODO
features = base_features + extra_features

df_model = df_clean.dropna(subset=features + ["disease"]).copy()
df_model.shape


(303, 15)

## Baseline model


In [5]:
X_base = sm.add_constant(df_model[base_features])
y = df_model["disease"]
m_base = sm.Logit(y, X_base).fit(disp=False)
print(m_base.summary())


                           Logit Regression Results                           
Dep. Variable:                disease   No. Observations:                  303
Model:                          Logit   Df Residuals:                      300
Method:                           MLE   Df Model:                            2
Date:                Tue, 19 May 2026   Pseudo R-squ.:                  0.1093
Time:                        14:34:24   Log-Likelihood:                -186.15
converged:                       True   LL-Null:                       -208.99
Covariance Type:            nonrobust   LLR p-value:                 1.206e-10
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -4.8077      0.898     -5.353      0.000      -6.568      -3.047
age            0.0657      0.015      4.427      0.000       0.037       0.095
sex            1.4989      0.289      5.179      0.0

## Extended model


In [6]:
X_ext = sm.add_constant(df_model[features])
m_ext = sm.Logit(y, X_ext).fit(disp=False)
print(m_ext.summary())


                           Logit Regression Results                           
Dep. Variable:                disease   No. Observations:                  303
Model:                          Logit   Df Residuals:                      297
Method:                           MLE   Df Model:                            5
Date:                Tue, 19 May 2026   Pseudo R-squ.:                  0.3187
Time:                        14:34:28   Log-Likelihood:                -142.39
converged:                       True   LL-Null:                       -208.99
Covariance Type:            nonrobust   LLR p-value:                 4.975e-27
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.4466      1.835     -0.243      0.808      -4.042       3.149
age            0.0328      0.018      1.795      0.073      -0.003       0.069
sex            1.4251      0.335      4.252      0.0

## Odds ratios + 95% CI


In [7]:
def odds_ratios_with_ci(model):
    params = model.params
    se = model.bse
    or_ = np.exp(params)
    lo = np.exp(params - 1.96*se)
    hi = np.exp(params + 1.96*se)
    return pd.DataFrame({"odds_ratio": or_, "ci_low": lo, "ci_high": hi, "p_value": model.pvalues})

display(odds_ratios_with_ci(m_base))
display(odds_ratios_with_ci(m_ext))


,odds_ratio,ci_low,ci_high,p_value
const,0.008166,0.001404,0.047485,8.656902e-08
age,1.067919,1.037297,1.099444,9.555449e-06
sex,4.476637,2.538692,7.893937,2.227803e-07


,odds_ratio,ci_low,ci_high,p_value
const,0.639811,0.017556,23.316942,0.807675
age,1.033376,0.996980,1.071101,0.072703
sex,4.158249,2.155707,8.021054,0.000021
thalach,0.976197,0.961480,0.991140,0.001882
oldpeak,1.869560,1.404434,2.488728,0.000018
exang,4.335065,2.266676,8.290902,0.000009


## TODO - Interpret two coefficients
- Variable 1: thalach (maximum heart rate achieved)
  - Interpretation: In the extended model, thalach has an odds ratio < 1 (OR ≈ 0.97).
  This means that for each additional beat per minute of maximum heart rate,
  the odds of having heart disease decrease by approximately 3%, holding age,
  sex, oldpeak, and exang constant.
  The 95% CI does not cross 1 and p < 0.001, so this effect is statistically
  significant and not explained by the other variables in the model.
  - What it does NOT prove:This does not prove that a lower max heart rate *causes* heart disease.
  The relationship may be reversed - disease impairs cardiac function and
  thereby reduces max heart rate. This is a cross-sectional dataset with no
  time ordering, so causality cannot be established.


- Variable 2: oldpeak (ST depression induced by exercise)
  - Interpretation: oldpeak has an odds ratio > 1 (OR ≈ 1.5-2.0).
  For each 1 mm increase in ST depression relative to rest, the odds of
  heart disease increase by approximately 50-100%, after controlling for
  age, sex, thalach, and exang.
  This is one of the strongest predictors in the model - both statistically
  (p < 0.001) and in terms of effect size.

  - What it does NOT prove:  A higher oldpeak value does not prove the patient has coronary artery
  disease. ST depression can result from other causes (left ventricular
  hypertrophy, medication effects, electrolyte imbalances). The model
  captures a statistical association, not a diagnostic rule.


Unmeasured confounder you worry about:

- Physical fitness level is not recorded in this dataset. Fit patients tend to
achieve higher max heart rates AND have lower disease risk - both independently.
Without controlling for fitness, the thalach effect may partly reflect a
fitness advantage rather than a direct cardiac mechanism. This could inflate
the apparent protective effect of high thalach in the model.